# Zaawansowany Pipeline EEG: Preprocessing + Savitzky-Golay & Gauss
Ten notebook zawiera najbardziej precyzyjną implementację przygotowania danych pod model **AtomVQVAE**.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter
import os

# Konfiguracja ścieżek
EDF_PATH = 'dane/Alpha1_raw.edf'
CSV_PATH = 'dane/NBackExample_imp.csv'

print("Biblioteki gotowe.")

## 1. Preprocessing Standardowy (Filtry + CAR + ICA)
Ładujemy dane, mapujemy kanały 10-20, stosujemy CAR i usuwamy artefakty oczne przez ICA.

In [ ]:
raw = mne.io.read_raw_edf(EDF_PATH, preload=True, verbose=False)

# Mapowanie nazw kanałów
mapping = {ch: ch.split('-')[0].split(':')[0].strip() for ch in raw.ch_names}
raw.rename_channels(mapping)

# Ustawienie montażu standardowego 10-20
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='ignore')

# Wybranie tylko kanałów EEG
raw.pick_types(eeg=True)

# Filtrowanie pasmowe 1-40 Hz i wycinające 50 Hz
raw.filter(1.0, 40.0, verbose=False)
raw.notch_filter(np.arange(50, 151, 50), verbose=False)

# Zmiana referencji na uśrednioną (CAR)
raw.set_eeg_reference('average', projection=False, verbose=False)

# ICA - Usuwanie mrugnięć
ica = mne.preprocessing.ICA(n_components=15, random_state=97, method='fastica', verbose=False)
ica.fit(raw)

# Automatyczne wyszukiwanie komponentów EOG na bazie Fp1/Fp2
eog_indices, _ = ica.find_bads_eog(raw, ch_name=['Fp1', 'Fp2'], verbose=False)
ica.exclude = eog_indices

raw_clean = raw.copy()
ica.apply(raw_clean, verbose=False)

print(f"ICA zakończone. Wykluczono {len(ica.exclude)} komponentów.")

## 2. Wygładzanie Sygnału: Filtr Savitzky-Golay
Stosujemy go **po ICA**, aby wygładzić sygnał w dziedzinie czasu. Jest to idealny filtr dla EEG, ponieważ zachowuje kształt fali (np. piki ERP).

In [ ]:
data = raw_clean.get_data()

# Parametry Savitzky-Golay:
# window_length: 15 (ok. 50ms przy 300Hz) - musi być nieparzyste
# polyorder: 2 (wielomian kwadratowy)
data_savgol = savgol_filter(data, window_length=15, polyorder=2, axis=1)

# Aktualizacja danych w obiekcie MNE (z zachowaniem struktury)
raw_savgol = raw_clean.copy()
raw_savgol._data = data_savgol

print("Zastosowano filtr Savitzky-Golay (window=15, poly=2).")

# Wizualizacja porównawcza
start, stop = 0, 600
plt.figure(figsize=(15, 6))
plt.plot(raw_clean.times[start:stop], data[0, start:stop] * 1e6, label='Po ICA (Surowy)', alpha=0.5, color='gray')
plt.plot(raw_savgol.times[start:stop], data_savgol[0, start:stop] * 1e6, label='Po Savitzky-Golay', color='blue', linewidth=1.5)
plt.xlabel("Czas [s]")
plt.ylabel("Amplituda [uV]")
plt.legend()
plt.title("Wygładzanie Savitzky-Golay na kanale " + raw_clean.ch_names[0])
plt.grid(True)
plt.show()

## 3. Analiza Częstotliwościowa i Wygładzanie Gaussowskie (Obraz)
Generujemy spektrogram (PSD Welch) i nakładamy rozmycie Gaussa, aby przygotować "czysty" obraz dla VQ-VAE.

In [ ]:
# Podział na 2-sekundowe epoki
epochs = mne.make_fixed_length_epochs(raw_savgol, duration=2.0, preload=True, verbose=False)

# Obliczanie PSD metodą Welcha (uśrednianie segmentów dla stabilności)
spectrum = epochs.compute_psd(method='welch', fmin=1, fmax=40, n_fft=256, verbose=False)
psd_data = spectrum.get_data()[0] # Pierwsza epoka, kształt: (kanały, częstotliwości)

# Logarytmowanie i normalizacja min-max do zakresu [0, 1]
img_raw = np.log10(psd_data)
img_norm = (img_raw - img_raw.min()) / (img_raw.max() - img_raw.min())

# Wygładzanie Gaussowskie (sigma=0.7 - delikatne rozmycie szumu pikselowego)
img_gauss = gaussian_filter(img_norm, sigma=0.7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
im1 = ax1.imshow(img_norm, aspect='auto', cmap='magma', interpolation='none')
ax1.set_title("Znormalizowany Spektrogram (Surowy)")
ax1.set_xlabel("Częstotliwość (Index)")
ax1.set_ylabel("Kanał (Index)")
plt.colorbar(im1, ax=ax1, label='Normalized Power')

im2 = ax2.imshow(img_gauss, aspect='auto', cmap='magma', interpolation='none')
ax2.set_title("Spektrogram po wygładzaniu Gaussa (sigma=0.7)")
ax2.set_xlabel("Częstotliwość (Index)")
plt.colorbar(im2, ax=ax2, label='Normalized Power')
plt.tight_layout()
plt.show()

## 4. Finalna Topomapa i Walidacja
Czysty rozkład mocy Alpha (8-12 Hz) po wszystkich stopniach przetwarzania.

In [ ]:
print("Końcowe pasma mocy (Topomaps):")
spectrum.plot_topomap(bands={'Alpha (8-12 Hz)': (8, 12), 'Beta (13-30 Hz)': (13, 30)}, ch_type='eeg', cmaps='RdBu_r');